# Enable Prefix Aware Routing on Amazon SageMaker Inference

Prefix aware routing is a new routing strategy that looks at the beginning of each request and consistently sends requests with the same beginning to the same instance.

In our benchmarks on Llama 3.1 70B, this reduced P50 TTFT by up to 77%, pushed KV cache hit rates from roughly 25% to over 80%, and increased throughput by up to 16%.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves 'Creating' or "Updating" state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [37]:
instance = {"type": "ml.g6e.4xlarge", "num_gpu": 1}

model_id = "Qwen/Qwen3.5-4B"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 600
variant_name = "v1"

In [38]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.26.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
}

## Deployment

In this section, we demonstrate how to enable PREFIX AWARE routing for both single model endpoints and Inference Component endpoints.

#### PLEASE RUN "SME ENDPOINT" or "IC ENDPOINT" cells.


Please note `RoutingConfig` entry below.

Two parameters control the behavior:

`PrefixLength` (valid values are 1024 to 65536): How much of the request to use for routing. For the native SageMaker Invoke API, this is bytes from the beginning of the request body. For the OpenAI-compatible API, this is characters from the extracted message text. Set this to cover your shared prefix plus enough unique content to spread different workloads across instances.

`ConcurrencyThreshold` (valid values are 1 to 1024): The maximum in-flight requests on the target instance before overflow kicks in. If the target instance is at this limit, the request goes to a less loaded instance instead. This is overload protection mechanism. If one prefix is extremely popular and the target instance is already at capacity, the endpoint routes the request to a less busy instance instead. You configure the concurrency limit, and the endpoint respects it. You might miss a cache hit on that one request, but you avoid overwhelming a single machine.

### SME endpoint

In [39]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 2,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "RoutingConfig": {
                "RoutingStrategy": "PREFIX_AWARE",
                "PrefixAwareRoutingConfig": {
                    "PrefixLength": 1024,
                      "ConcurrencyThreshold": 10
                }
            }
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

In [41]:
#
# Helper function for SME
#
ic_name = None

def invoke_endpoint(payload, endpoint_name=endpoint_name, print_output=False, reasoning_tag="reasoning"):
    start_time = time.time()
    res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                     Body=json.dumps(payload),
                                     ContentType="application/json")
    response = json.loads(res["Body"].read().decode("utf8"))
    end_time = time.time()

    if print_output: print(f"✅ Response time: {end_time-start_time:.2f}s\n")

    output = ""
    reasoning = response["choices"][0]["message"].get(reasoning_tag, None)
    if reasoning:
        output += f"### Reasoning:\n---\n{reasoning}\n\n"

    content = response["choices"][0]["message"].get("content", None)
    if content:
        output += f"### Content:\n---\n{content}\n\n---"

    if print_output: display(Markdown(output))

    return response

#### Invoke_Endpoint example

In [43]:
payload={"messages": [{"role": "user", "content": "Who are you?"}]}

_ = invoke_endpoint(payload, endpoint_name, True)

#### OpenAI API example

In [44]:
%pip install openai --quiet --no-warn-conflicts

In [46]:
from openai import OpenAI
from sagemaker.core.token_generator import generate_token
sme_base_url = f"https://runtime.sagemaker.{region}.amazonaws.com/endpoints/{endpoint_name}/openai/v1"
client = OpenAI(
    base_url=sme_base_url,
    api_key=generate_token(region=region)
)

print(f"Base URL: {sme_base_url}\n")

In [47]:
response = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who are you?"},
    ],
    stream=False,
)

print(f"Response:      {response.choices[0].message.content}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Token usage:   {response.usage}")

### IC endpoint

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 2,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "RoutingConfig": {
                "RoutingStrategy": "PREFIX_AWARE",
                "PrefixAwareRoutingConfig": {
                    "PrefixLength": 1024,
                      "ConcurrencyThreshold": 10
                }
            }
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

In [ ]:
compute_req = {
    "MinMemoryRequiredInMb": 512,
    "NumberOfAcceleratorDevicesRequired": 1,
}

ic_name = f"ic-{model_name}"

_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
        "ComputeResourceRequirements": compute_req,
    },
    RuntimeConfig={
        "CopyCount": 2,
    },
)
_ = wait_for_ic(ic_name)

In [ ]:
#
# Helper function
#
def invoke_endpoint(payload, endpoint_name=endpoint_name, ic_name=ic_name, print_output=False, reasoning_tag="reasoning"):
    start_time = time.time()
    res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                     InferenceComponentName=ic_name,
                                     Body=json.dumps(payload),
                                     ContentType="application/json")
    #print(res)
    response = json.loads(res["Body"].read().decode("utf8"))
    end_time = time.time()

    if print_output: print(f"✅ Response time: {end_time-start_time:.2f}s\n")

    output = ""
    reasoning = response["choices"][0]["message"].get(reasoning_tag, None)
    if reasoning:
        output += f"### Reasoning:\n---\n{reasoning}\n\n"

    content = response["choices"][0]["message"].get("content", None)
    if content:
        output += f"### Content:\n---\n{content}\n\n---"

    if print_output: display(Markdown(output))

    return response

## Inference Examples

---
### Simple performance test

In [ ]:
import time
import numpy as np

def run_perf_test(llm, num_iterations, payload):
    results = []
    for i in range(num_iterations):
        start = time.time()
        _ = invoke_endpoint(payload=payload)
        results.append((time.time() - start) * 1000)

    print("\nPrediction latency: \n")
    print("P95: " + str(np.percentile(results, 95)) + " ms")
    print("P90: " + str(np.percentile(results, 90)) + " ms")
    print("Average: " + str(np.average(results)) + " ms")

In [ ]:
#
# Calculate runtime performance
#
payload = {
      "messages": [
          {
              "role": "user",
              "content": """Summarize the following text:
  In 2043, Mira repaired obsolete service robots in a quiet coastal town. One rainy evening, a battered archive unit arrived with a cracked display and a single request: help me
  remember why I was built. Its memory held weather reports, old recipes, and recordings of children asking questions. Mira restored the files one by one. The unit began connecting
  forgotten details: a fisherman who checked on his neighbor, a teacher who saved every drawing, a nurse who hummed during long nights. It had been designed to preserve local
  history, but years of damage had left its purpose unclear. Mira gave it a new solar cell and carried it to the town library. There, the AI organized stories for anyone who asked.
  It never claimed to understand people. Instead, it listened carefully, returned what they had shared, and helped them see that small acts of care could outlast the storms for
  many years ahead."""
          }
      ],
  }
run_perf_test(llm=endpoint_name, num_iterations=20, payload=payload)

In CloudWatch log you will that during the first request, the model container processed the prompt:

`Avg prompt throughput: 21.2 tokens/s, Avg generation throughput: 78.5 tokens/s`

But in all subsequent message, the prompt throughput is 0 because the requests are send to the same instance and prompt is extracted from the KV-cache withough reprocessing:

`Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 79.4 tokens/s`

## Cleanup

In [48]:
if ic_name:
    _ = sm.delete_inference_component(InferenceComponentName=ic_name)

In [49]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)